In [ ]:
# ============================================================
# PatchCore Essential Edge Metrics | Local Copy + Compute/End-to-End Timing with Top-k Mean Image Scoring
# Dataset: Lusitano_Dataset
# SEED = 42 | No CenterCrop
# ============================================================

import os, gc, time, random, shutil, psutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b5, EfficientNet_B5_Weights
from PIL import Image, ImageFile

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# ============================================================
# 1) Reproducibility
# ============================================================

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

ImageFile.LOAD_TRUNCATED_IMAGES = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# ============================================================
# 2) Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# 3) Dataset paths: copy from Google Drive to local Colab storage
# ============================================================

COPY_DATASET_TO_LOCAL = True
DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/<YOUR_DATASET_FOLDER>/Lusitano_Dataset")
LOCAL_DATASET_ROOT = Path("/content/Lusitano_Dataset")

if COPY_DATASET_TO_LOCAL:
    if not DRIVE_DATASET_ROOT.exists():
        raise ValueError(f"Drive dataset not found: {DRIVE_DATASET_ROOT}")

    expected_local_train = LOCAL_DATASET_ROOT / "nondefects" / "nondefects"
    expected_local_test = LOCAL_DATASET_ROOT / "test" / "test"

    if expected_local_train.exists() and expected_local_test.exists():
        print("Local dataset already exists:", LOCAL_DATASET_ROOT)
    else:
        if LOCAL_DATASET_ROOT.exists():
            print("Removing incomplete local dataset copy...")
            shutil.rmtree(LOCAL_DATASET_ROOT)

        print("Copying dataset from Google Drive to local Colab storage...")
        print("Source:", DRIVE_DATASET_ROOT)
        print("Target:", LOCAL_DATASET_ROOT)
        copy_start = time.time()
        shutil.copytree(DRIVE_DATASET_ROOT, LOCAL_DATASET_ROOT)
        print(f"Dataset copy completed in {(time.time() - copy_start) / 60:.2f} minutes.")

    DATASET_ROOT = LOCAL_DATASET_ROOT
else:
    DATASET_ROOT = DRIVE_DATASET_ROOT

print("Using DATASET_ROOT:", DATASET_ROOT)

train_good_path = DATASET_ROOT / "nondefects" / "nondefects"
test_root_path  = DATASET_ROOT / "test" / "test"

if not train_good_path.exists():
    raise ValueError(f"Training path not found: {train_good_path}")

if not test_root_path.exists():
    raise ValueError(f"Test path not found: {test_root_path}")

print("Train folder:", train_good_path)
print("Test folder :", test_root_path)

# ============================================================
# 4) Fixed experiment settings
# Keep these identical for Reservoir and Greedy Coreset
# ============================================================

IMG_SIZE = 448
BATCH_SIZE = 16
NUM_WORKERS = 0  # local /content storage; set 0 if Colab worker issues occur

PATCHES_PER_IMAGE = 200
PRE_POOL = 400_000
MAX_MEM_PATCHES = 20_000

NN_CHUNK = 40_000
THRESH_SAMPLE_IMAGES = 2000

# Top-k mean image scoring
# Image score = mean of top 1% largest nearest-neighbor patch distances
SCORE_MODE = "topk_mean"
TOPK_FRAC = 0.01

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ============================================================
# 5) Metric / memory helpers
# ============================================================

def bytes_to_mb(x):
    return x / (1024 ** 2)

def tensor_size_mb(tensor):
    return bytes_to_mb(tensor.numel() * tensor.element_size())

def model_size_mb(model):
    total_bytes = 0

    for p in model.parameters():
        total_bytes += p.numel() * p.element_size()

    for b in model.buffers():
        total_bytes += b.numel() * b.element_size()

    return bytes_to_mb(total_bytes)

def current_ram_mb():
    process = psutil.Process(os.getpid())
    return bytes_to_mb(process.memory_info().rss)

def reset_peak_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def get_peak_memory_mb():
    if torch.cuda.is_available():
        return bytes_to_mb(torch.cuda.max_memory_allocated())
    else:
        return current_ram_mb()

# ============================================================
# 6) Transform
# No CenterCrop is used
# ============================================================

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# ============================================================
# 7) Dataset utilities
# ============================================================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def list_images(folder: Path, verify_images=False):
    """
    Fast image listing. verify_images=False avoids slow Image.open(...).verify()
    over Google Drive/local folders. This does not affect AP/F1/AUC if the
    dataset has already been validated in previous runs.
    """
    valid_paths = []

    for p in sorted(folder.glob("*")):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            if verify_images:
                try:
                    Image.open(p).verify()
                    valid_paths.append(p)
                except Exception as e:
                    print("Corrupted image skipped:", p, e)
            else:
                valid_paths.append(p)

    return valid_paths

class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert("RGB")
        return self.transform(img), str(p)

class TestImageDataset(Dataset):
    def __init__(self, items, transform):
        self.items = list(items)
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        p, label = self.items[idx]
        img = Image.open(p).convert("RGB")
        return self.transform(img), label, str(p)

# ============================================================
# 8) Feature extractor: EfficientNet-B5
# ============================================================

class EfficientNetFeatureExtractor(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.model = efficientnet_b5(weights=EfficientNet_B5_Weights.DEFAULT)
        self.model.eval()

        for p in self.model.parameters():
            p.requires_grad = False

        self.features = []

        def hook(_, __, output):
            self.features.append(output)

        self.model.features[3].register_forward_hook(hook)
        self.model.features[5].register_forward_hook(hook)
        self.model.features[7].register_forward_hook(hook)

    def forward(self, x):
        self.features = []

        with torch.no_grad():
            _ = self.model(x)

        if len(self.features) == 0:
            raise RuntimeError("No features extracted. Check hook registration.")

        fmap_size = min(f.shape[-2] for f in self.features)
        resize = torch.nn.AdaptiveAvgPool2d(fmap_size)

        resized = [resize(f) for f in self.features]
        patch_features = torch.cat(resized, dim=1)

        B, C, H, W = patch_features.shape

        patch_features = patch_features.reshape(B, C, H * W).permute(0, 2, 1)

        return patch_features  # (B, N, C)

# ============================================================
# 9) Load training images and backbone
# ============================================================

train_paths = list_images(train_good_path)

if len(train_paths) == 0:
    raise ValueError("No training images found.")

print("Training normal images:", len(train_paths))

train_loader = DataLoader(
    ImagePathDataset(train_paths, transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
    drop_last=False
)

backbone = EfficientNetFeatureExtractor().to(DEVICE).eval()

# ============================================================
# 10) Build common candidate patch pool
# ============================================================

set_seed(SEED)

all_features = []

print("\nExtracting candidate patch pool...")

for xb, _ in tqdm(train_loader):
    xb = xb.to(DEVICE, non_blocking=True)

    with torch.no_grad():
        feats = backbone(xb)  # (B, N, C)

    B, N, C = feats.shape

    for i in range(B):
        n = min(PATCHES_PER_IMAGE, N)
        idx = torch.randperm(N, device=DEVICE)[:n]
        sampled = feats[i, idx].detach().float().cpu()
        all_features.append(sampled)

    del xb, feats

candidate_pool = torch.cat(all_features, dim=0)
del all_features

print("Candidate pool before PRE_POOL cap:", candidate_pool.shape)

if candidate_pool.shape[0] > PRE_POOL:
    set_seed(SEED)
    idx = torch.randperm(candidate_pool.shape[0])[:PRE_POOL]
    candidate_pool = candidate_pool[idx]

print("Final candidate pool:", candidate_pool.shape)

# ============================================================
# 11) Memory-bank selection: Greedy Coreset
# ============================================================

METHOD_NAME = "PatchCore + Greedy Coreset + Top-k Mean"
CORESET_CHUNK = 40_000

@torch.no_grad()
def greedy_coreset_gpu(
    features_cpu,
    max_samples,
    chunk=40_000,
    use_fp16=True,
    device="cuda",
    seed=42
):
    random.seed(seed)

    N, C = features_cpu.shape

    if N <= max_samples:
        return features_cpu.clone()

    feats = features_cpu.to(device, non_blocking=True).contiguous()

    if use_fp16:
        feats = feats.half()

    selected_idx = torch.empty((max_samples,), dtype=torch.long, device=device)

    first = random.randint(0, N - 1)
    selected_idx[0] = first

    center = feats[first:first+1]

    min_d = torch.empty((N,), device=device, dtype=torch.float32)

    for start in range(0, N, chunk):
        x = feats[start:start+chunk]
        d = (x - center).float().pow(2).sum(dim=1)
        min_d[start:start+chunk] = d

    for i in tqdm(range(1, max_samples), desc="Greedy coreset"):
        farthest = torch.argmax(min_d).item()
        selected_idx[i] = farthest

        center = feats[farthest:farthest+1]

        for start in range(0, N, chunk):
            x = feats[start:start+chunk]
            d = (x - center).float().pow(2).sum(dim=1)
            min_d[start:start+chunk] = torch.minimum(
                min_d[start:start+chunk],
                d
            )

    selected = feats[selected_idx].float().cpu()

    del feats, selected_idx, min_d

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return selected

print("\nBuilding greedy coreset memory bank...")

memory_bank_cpu = greedy_coreset_gpu(
    candidate_pool,
    max_samples=MAX_MEM_PATCHES,
    chunk=CORESET_CHUNK,
    use_fp16=True,
    device=DEVICE,
    seed=SEED
)

del candidate_pool

# ============================================================
# 12) Essential model / memory footprint
# ============================================================

memory_bank_size_mb = tensor_size_mb(memory_bank_cpu)
backbone_size_mb = model_size_mb(backbone)
estimated_total_footprint_mb = memory_bank_size_mb + backbone_size_mb

print(f"Memory-bank size MB: {memory_bank_size_mb:.2f}")
print(f"Backbone/model size MB: {backbone_size_mb:.2f}")
print(f"Estimated total footprint MB: {estimated_total_footprint_mb:.2f}")

memory_bank_gpu = memory_bank_cpu.to(DEVICE, non_blocking=True)

# ============================================================
# 13) Image-level anomaly score: Top-k Mean
# Patch score = nearest-neighbor distance to memory bank
# Image score = mean of top k% highest patch scores
# ============================================================

@torch.no_grad()
def image_anomaly_score(patch_feats_gpu, memory_bank_gpu, chunk_size=40_000):
    x = patch_feats_gpu.float()
    P = x.shape[0]

    min_dist = torch.full(
        (P,),
        float("inf"),
        device=DEVICE,
        dtype=torch.float32
    )

    x2 = x.pow(2).sum(dim=1, keepdim=True)

    for start in range(0, memory_bank_gpu.shape[0], chunk_size):
        mb = memory_bank_gpu[start:start + chunk_size].float()
        mb2 = mb.pow(2).sum(dim=1).unsqueeze(0)

        dot = x @ mb.t()
        d2 = x2 + mb2 - 2.0 * dot
        d2 = torch.clamp(d2, min=0.0)

        min_dist = torch.minimum(min_dist, d2.min(dim=1).values)

    patch_scores = min_dist.sqrt()

    if SCORE_MODE == "max":
        return patch_scores.max().item()

    elif SCORE_MODE == "topk_mean":
        k = max(1, int(patch_scores.numel() * TOPK_FRAC))
        topk_scores = torch.topk(patch_scores, k=k, largest=True).values
        return topk_scores.mean().item()

    else:
        raise ValueError(f"Unknown SCORE_MODE: {SCORE_MODE}")

# ============================================================
# 14) Threshold for F1 calculation
# This threshold is used internally but saved for reproducibility.
# ============================================================

rng = np.random.default_rng(SEED + 100)

num_for_thresh = min(THRESH_SAMPLE_IMAGES, len(train_paths))
thresh_indices = rng.permutation(len(train_paths))[:num_for_thresh]
thresh_subset = [train_paths[i] for i in thresh_indices]

thresh_loader = DataLoader(
    ImagePathDataset(thresh_subset, transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
    drop_last=False
)

threshold_scores = []

print("\nComputing threshold from normal training subset...")

for xb, _ in tqdm(thresh_loader, desc="Threshold"):
    xb = xb.to(DEVICE, non_blocking=True)

    with torch.no_grad():
        feats = backbone(xb)

    for i in range(feats.shape[0]):
        score = image_anomaly_score(
            feats[i],
            memory_bank_gpu,
            chunk_size=NN_CHUNK
        )
        threshold_scores.append(score)

    del xb, feats

threshold_scores = np.array(threshold_scores)

threshold = threshold_scores.mean() + 3.0 * threshold_scores.std(ddof=1)

print(f"Internal threshold for F1: {threshold:.6f}")

# ============================================================
# 15) Prepare test set
# ============================================================

def get_label(folder_name):
    name = folder_name.lower().replace("_", "-").strip()

    if name == "non-defects":
        return 0

    if name == "defects":
        return 1

    return None

test_items = []

for folder in sorted(test_root_path.iterdir()):
    if not folder.is_dir():
        continue

    label = get_label(folder.name)

    if label is None:
        print("Skipping unknown folder:", folder.name)
        continue

    paths = list_images(folder)

    for p in paths:
        test_items.append((p, label))

if len(test_items) == 0:
    raise ValueError("No test images found.")

print("\nTotal test images:", len(test_items))
print("Normal test images:", sum(1 for _, y in test_items if y == 0))
print("Defect test images:", sum(1 for _, y in test_items if y == 1))

test_loader = DataLoader(
    TestImageDataset(test_items, transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
    drop_last=False
)

# ============================================================
# 16) Test evaluation + inference time + peak memory usage
# ============================================================

reset_peak_memory()

if torch.cuda.is_available():
    torch.cuda.synchronize()

end_to_end_start = time.perf_counter()
compute_total_time = 0.0

y_true = []
y_score = []

print("\nEvaluating test set...")

for xb, yb, _ in tqdm(test_loader, desc="Testing"):
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    compute_start = time.perf_counter()

    xb = xb.to(DEVICE, non_blocking=True)

    with torch.no_grad():
        feats = backbone(xb)

    for i in range(feats.shape[0]):
        score = image_anomaly_score(
            feats[i],
            memory_bank_gpu,
            chunk_size=NN_CHUNK
        )

        y_score.append(score)
        y_true.append(int(yb[i]))

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    compute_total_time += time.perf_counter() - compute_start

    del xb, feats

if torch.cuda.is_available():
    torch.cuda.synchronize()

end_to_end_total_time = time.perf_counter() - end_to_end_start
compute_inference_time_per_image = compute_total_time / len(y_true)
end_to_end_local_runtime_per_image = end_to_end_total_time / len(y_true)
peak_memory_usage_mb = get_peak_memory_mb()

# ============================================================
# 17) Essential metrics
# ============================================================

y_true = np.array(y_true)
y_score = np.array(y_score)

y_pred = (y_score > threshold).astype(int)

auc_roc = roc_auc_score(y_true, y_score)
map_ap = average_precision_score(y_true, y_score)
f1 = f1_score(y_true, y_pred)

# ============================================================
# 18) Final result table
# ============================================================

result = {
    "Method": METHOD_NAME,
    "Seed": SEED,
    "Dataset_Root_Used": str(DATASET_ROOT),
    "IMG_SIZE": IMG_SIZE,
    "BATCH_SIZE": BATCH_SIZE,
    "NUM_WORKERS": NUM_WORKERS,
    "Score_Mode": SCORE_MODE,
    "TopK_Frac": TOPK_FRAC,
    "Internal_Threshold": threshold,
    "AUC_ROC": auc_roc,
    "mAP_AP": map_ap,
    "F1_Score": f1,
    "Inference_Time_Per_Image_sec": compute_inference_time_per_image,
    "Compute_Inference_Time_Per_Image_sec": compute_inference_time_per_image,
    "End_To_End_Local_Runtime_Per_Image_sec": end_to_end_local_runtime_per_image,
    "Compute_Total_Time_sec": compute_total_time,
    "End_To_End_Local_Total_Time_sec": end_to_end_total_time,
    "Memory_Bank_Size_MB": memory_bank_size_mb,
    "Backbone_Model_Size_MB": backbone_size_mb,
    "Estimated_Total_Footprint_MB": estimated_total_footprint_mb,
    "Peak_Memory_Usage_MB": peak_memory_usage_mb
}

df_result = pd.DataFrame([result])

print("\n==============================")
print("ESSENTIAL EDGE COMPARISON RESULT")
print("==============================")
display(df_result)

# ============================================================
# 19) Save result
# ============================================================

save_dir = "/content/drive/MyDrive/<YOUR_OUTPUT_FOLDER>/localcopy_compute_timing"
os.makedirs(save_dir, exist_ok=True)

safe_method_name = METHOD_NAME.lower().replace(" ", "_").replace("+", "plus")
save_path = f"{save_dir}/{safe_method_name}_essential_edge_metrics_seed42.csv"

df_result.to_csv(save_path, index=False)

print(f"\nSaved result to: {save_path}")